# Customer Shopping Behavior: SQL Analysis

Loads the cleaned dataset into an in-memory SQLite database and answers business questions with SQL.

In [11]:
import sqlite3
import pandas as pd

# Local path (repo layout). In Colab, use:
# '/content/drive/MyDrive/data/customer_shopping_behavior_cleaned.csv'
CSV_PATH = '../data/processed/customer_shopping_behavior_cleaned.csv'

df = pd.read_csv(CSV_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('customer', conn, index=False, if_exists='replace')

def run(query):
    return pd.read_sql(query, conn)

print(df.shape)
run('SELECT * FROM customer LIMIT 5')

(3900, 19)


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365


## 1. Overall KPIs

In [12]:
run("""
SELECT COUNT(*) AS customers,
       SUM(purchase_amount) AS total_revenue,
       ROUND(AVG(purchase_amount), 2) AS avg_spend,
       ROUND(AVG(review_rating), 2) AS avg_rating
FROM customer
""")

,customers,total_revenue,avg_spend,avg_rating
0,3900,233081,59.76,3.75


## 2. Revenue by gender

In [13]:
run("""
SELECT gender,
       COUNT(*) AS customers,
       SUM(purchase_amount) AS revenue
FROM customer
GROUP BY gender
ORDER BY revenue DESC
""")

,gender,customers,revenue
0,Male,2652,157890
1,Female,1248,75191


## 3. Top 5 products by average review rating

In [14]:
run("""
SELECT item_purchased,
       ROUND(AVG(review_rating), 2) AS avg_rating,
       COUNT(*) AS purchases
FROM customer
GROUP BY item_purchased
ORDER BY avg_rating DESC
LIMIT 5
""")

,item_purchased,avg_rating,purchases
0,Gloves,3.86,140
1,Sandals,3.84,160
2,Boots,3.82,144
3,Hat,3.80,154
4,T-shirt,3.78,147


## 4. Subscribers vs non-subscribers

In [15]:
run("""
SELECT subscription_status,
       COUNT(*) AS customers,
       ROUND(AVG(purchase_amount), 2) AS avg_spend,
       SUM(purchase_amount) AS revenue
FROM customer
GROUP BY subscription_status
""")

,subscription_status,customers,avg_spend,revenue
0,No,2847,59.87,170436
1,Yes,1053,59.49,62645


## 5. Revenue by age group

In [16]:
run("""
SELECT age_group,
       COUNT(*) AS customers,
       SUM(purchase_amount) AS revenue
FROM customer
GROUP BY age_group
ORDER BY CASE age_group
           WHEN 'Young Adult' THEN 1
           WHEN 'Adult' THEN 2
           WHEN 'Middle-aged' THEN 3
           WHEN 'Senior' THEN 4
         END
""")

,age_group,customers,revenue
0,Young Adult,1028,62143
1,Adult,942,55978
2,Middle-aged,986,59197
3,Senior,944,55763


## 6. Revenue by category

In [17]:
run("""
SELECT category,
       SUM(purchase_amount) AS revenue,
       ROUND(AVG(purchase_amount), 2) AS avg_spend
FROM customer
GROUP BY category
ORDER BY revenue DESC
""")

,category,revenue,avg_spend
0,Clothing,104264,60.03
1,Accessories,74200,59.84
2,Footwear,36093,60.26
3,Outerwear,18524,57.17


## 7. Revenue by season

In [18]:
run("""
SELECT season,
       SUM(purchase_amount) AS revenue,
       COUNT(*) AS purchases
FROM customer
GROUP BY season
ORDER BY revenue DESC
""")

,season,revenue,purchases
0,Fall,60018,975
1,Spring,58679,999
2,Winter,58607,971
3,Summer,55777,955


## 8. Discount vs no discount: average spend

In [19]:
run("""
SELECT discount_applied,
       COUNT(*) AS purchases,
       ROUND(AVG(purchase_amount), 2) AS avg_spend,
       SUM(purchase_amount) AS revenue
FROM customer
GROUP BY discount_applied
""")

,discount_applied,purchases,avg_spend,revenue
0,No,2223,60.13,133670
1,Yes,1677,59.28,99411


## 9. Top 10 states by revenue

In [ ]:
run("""
SELECT location,
       SUM(purchase_amount) AS revenue,
       COUNT(*) AS customers
FROM customer
GROUP BY location
ORDER BY revenue DESC
LIMIT 10
""")